# 02 · Python Data Preparation: MySQL Export & RFM Validation

**Project:** Olist E-Commerce Customer Segmentation & Retention Analysis  
## Purpose
This notebook bridges the Python and SQL stages of the project:
1. Reconciles the table imported into MySQL against the original CSV.
2. Exports the remaining tables as header-less CSV files for MySQL bulk loading (the large geolocation table is split into chunks).
3. Validates the RFM table produced by the SQL stage and saves a clean copy for the analysis notebooks.

| | |
|---|---|
| **Input** | Original Kaggle ZIP · `customers.csv` (imported copy) · `customer_rfm_segments.csv` (SQL output) |
| **Output** | Header-less CSVs for MySQL import · `customer_rfm_segments_clean.csv` |
| **Libraries** | pandas |
| **Environment** | Google Colab (outputs are written to `/content/`; download them from the Files panel or mount Google Drive) |

> **How to run:** execute the cells in order and upload the requested files when prompted.

## 1. Setup: Load and Verify the Dataset
Same setup as Notebook 01: extract the Kaggle ZIP and register the CSV paths used below.

In [ ]:

import os
import zipfile
import pandas as pd
from google.colab import files

# Upload the original Kaggle ZIP
uploaded = files.upload()

zip_path = next(iter(uploaded.keys()))

# Extract dataset
extract_path = "/content/olist_dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("Location:", extract_path)


Saving archive (5).zip to archive (5).zip
Dataset extracted successfully!
Location: /content/olist_dataset


In [ ]:

import os
import csv

expected_files = [
    "olist_customers_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv"
]

# Locate extracted CSV files
available_files = {}

for root, dirs, filenames in os.walk(extract_path):
    for filename in filenames:
        if filename.endswith(".csv"):
            available_files[filename] = os.path.join(root, filename)

print("DATASET VERIFICATION REPORT")
print("=" * 65)

for filename in expected_files:
    if filename in available_files:
        path = available_files[filename]

        with open(path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.reader(f)
            header = next(reader, [])
            row_count = sum(1 for _ in reader)

        size_mb = os.path.getsize(path) / (1024 ** 2)

        print(f"\nFile: {filename}")
        print(f"Rows: {row_count:,}")
        print(f"Columns: {len(header)}")
        print(f"Size: {size_mb:.2f} MB")
        print(f"Status: FOUND")
    else:
        print(f"\nFile: {filename}")
        print("Status: MISSING")


DATASET VERIFICATION REPORT

File: olist_customers_dataset.csv
Rows: 99,441
Columns: 5
Size: 8.62 MB
Status: FOUND

File: olist_orders_dataset.csv
Rows: 99,441
Columns: 8
Size: 16.84 MB
Status: FOUND

File: olist_order_items_dataset.csv
Rows: 112,650
Columns: 7
Size: 14.72 MB
Status: FOUND

File: olist_order_payments_dataset.csv
Rows: 103,886
Columns: 5
Size: 5.51 MB
Status: FOUND

File: olist_order_reviews_dataset.csv
Rows: 99,224
Columns: 7
Size: 13.78 MB
Status: FOUND

File: olist_products_dataset.csv
Rows: 32,951
Columns: 9
Size: 2.27 MB
Status: FOUND

File: olist_sellers_dataset.csv
Rows: 3,095
Columns: 4
Size: 0.17 MB
Status: FOUND

File: olist_geolocation_dataset.csv
Rows: 1,000,163
Columns: 5
Size: 58.44 MB
Status: FOUND

File: product_category_name_translation.csv
Rows: 71
Columns: 2
Size: 0.00 MB
Status: FOUND


## 2. Reconciliation of the MySQL Import
Compares the imported copy of the customers table (`customers.csv`) against the original file.

In [ ]:
customers_check = pd.read_csv(
    available_files["olist_customers_dataset.csv"]
)

print("CSV rows:", len(customers_check))
print("CSV columns:", customers_check.columns.tolist())

CSV rows: 99441
CSV columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [ ]:
import pandas as pd

original = pd.read_csv(
    available_files["olist_customers_dataset.csv"],
    dtype=str
)

imported = pd.read_csv(
    "/content/customers.csv",
    dtype=str
)

print("Original rows:", len(original))
print("Imported rows:", len(imported))

extra = imported.merge(
    original,
    how="left",
    indicator=True
)

extra = extra[extra["_merge"] == "left_only"]

print("Extra records:", len(extra))
display(extra)

Original rows: 99441
Imported rows: 99442
Extra records: 1


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,_merge
81038,customer_id,customer_unique_id,customer_z,customer_city,custo,left_only


#### Findings: Import reconciliation
The imported file has **99,442 rows** versus **99,441** in the original: **1 extra record**, which is the CSV header row (`customer_id`, `customer_unique_id`, …) stored as data.

To avoid this, the remaining tables below are exported **without header rows**.

## 3. Export Header-less CSVs for MySQL Bulk Loading
Each table is read with pandas and written back without its header row.

| Table | Rows exported |
|---|---:|
| orders | 99,441 |
| order_items | 112,650 |
| products | 32,951 |
| sellers | 3,095 |
| order_payments | 103,886 |
| order_reviews | 99,224 |
| geolocation | 1,000,163 (in 21 chunks of up to 50,000 rows) |
| category translation | 71 |

In [ ]:
import pandas as pd

orders_df = pd.read_csv(
    available_files["olist_orders_dataset.csv"]
)

orders_df.to_csv(
    "/content/orders_no_header.csv",
    index=False,
    header=False
)

print("Original rows:", len(orders_df))
print("Header removed successfully!")

Original rows: 99441
Header removed successfully!


In [ ]:
items_df = pd.read_csv(
    available_files["olist_order_items_dataset.csv"]
)

items_df.to_csv(
    "/content/order_items_no_header.csv",
    index=False,
    header=False
)

print("Total rows:", len(items_df))

Total rows: 112650


In [ ]:
products_df = pd.read_csv(
    available_files["olist_products_dataset.csv"]
)

products_df.to_csv(
    "/content/products_no_header.csv",
    index=False,
    header=False
)

print("Total products:", len(products_df))

Total products: 32951


In [ ]:
sellers_df = pd.read_csv(
    available_files["olist_sellers_dataset.csv"]
)

sellers_df.to_csv(
    "/content/sellers_no_header.csv",
    index=False,
    header=False
)

print("Total sellers:", len(sellers_df))

Total sellers: 3095


In [ ]:
payments_df = pd.read_csv(
    available_files["olist_order_payments_dataset.csv"]
)

payments_df.to_csv(
    "/content/order_payments_no_header.csv",
    index=False,
    header=False
)

print("Total payment records:", len(payments_df))

Total payment records: 103886


In [ ]:
reviews_df = pd.read_csv(
    available_files["olist_order_reviews_dataset.csv"]
)

reviews_df.to_csv(
    "/content/order_reviews_no_header.csv",
    index=False,
    header=False
)

print("Total review records:", len(reviews_df))

Total review records: 99224


### 3.1 Geolocation table (chunked export)
The geolocation table has over one million rows, so it is split into chunks of 50,000 rows to keep each file small enough to import.

In [ ]:
import pandas as pd
import os

geo_path = available_files["olist_geolocation_dataset.csv"]

chunks_folder = "/content/geolocation_chunks"
os.makedirs(chunks_folder, exist_ok=True)

chunk_size = 50000
chunk_number = 1
total_rows = 0

for chunk in pd.read_csv(geo_path, chunksize=chunk_size):

    output_path = os.path.join(
        chunks_folder,
        f"geolocation_part_{chunk_number}.csv"
    )

    chunk.to_csv(
        output_path,
        index=False,
        header=False
    )

    total_rows += len(chunk)

    print(
        f"Part {chunk_number}: {len(chunk)} rows"
    )

    chunk_number += 1

print("\nTotal chunks:", chunk_number - 1)
print("Total rows:", total_rows)

Part 1: 50000 rows
Part 2: 50000 rows
Part 3: 50000 rows
Part 4: 50000 rows
Part 5: 50000 rows
Part 6: 50000 rows
Part 7: 50000 rows
Part 8: 50000 rows
Part 9: 50000 rows
Part 10: 50000 rows
Part 11: 50000 rows
Part 12: 50000 rows
Part 13: 50000 rows
Part 14: 50000 rows
Part 15: 50000 rows
Part 16: 50000 rows
Part 17: 50000 rows
Part 18: 50000 rows
Part 19: 50000 rows
Part 20: 50000 rows
Part 21: 163 rows

Total chunks: 21
Total rows: 1000163


### 3.2 Category translation table

In [ ]:
from google.colab import files

df = pd.read_csv(
    available_files["product_category_name_translation.csv"]
)

df.to_csv(
    "/content/category_translation_no_header.csv",
    index=False,
    header=False
)

files.download("/content/category_translation_no_header.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Validation of the SQL-Generated RFM Table
The RFM table (`customer_rfm_segments.csv`) was produced in the SQL stage. Here it is loaded and checked for structure, score integrity and segment distribution before analysis.

### 4.1 Load the RFM table

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded.keys()))

rfm_df = pd.read_csv(file_name)

print("File loaded successfully!")
print("Dataset shape:", rfm_df.shape)

rfm_df.head()

Saving customer_rfm_segments.csv to customer_rfm_segments.csv
File loaded successfully!
Dataset shape: (93358, 10)


,customer_unique_id,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_score,RFM_total,customer_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,129.90,4,1,2,412,7,Potential Loyalists
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,18.90,4,1,1,411,6,Potential Loyalists
2,0000f46a3911fa3c0805444483337064,537,1,69.00,2,1,1,211,4,Hibernating
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,25.99,3,1,1,311,5,Needs Nurturing
4,0004aac84e0df4da2b147fca70cf8255,288,1,180.00,3,1,2,312,6,Needs Nurturing


### 4.2 Structure and data-type checks

In [ ]:
# Check column names and data types
print("DATA TYPES:")
print(rfm_df.dtypes)

print("\nMISSING VALUES:")
print(rfm_df.isnull().sum())

print("\nDUPLICATE ROWS:")
print(rfm_df.duplicated().sum())

print("\nDATASET INFORMATION:")
rfm_df.info()

DATA TYPES:
customer_unique_id     object
recency_days            int64
frequency               int64
monetary              float64
R_score                 int64
F_score                 int64
M_score                 int64
RFM_score               int64
RFM_total               int64
customer_segment       object
dtype: object

MISSING VALUES:
customer_unique_id    0
recency_days          0
frequency             0
monetary              0
R_score               0
F_score               0
M_score               0
RFM_score             0
RFM_total             0
customer_segment      0
dtype: int64

DUPLICATE ROWS:
0

DATASET INFORMATION:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93358 entries, 0 to 93357
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_unique_id  93358 non-null  object 
 1   recency_days        93358 non-null  int64  
 2   frequency           93358 non-null  int64  
 3   mone

In [ ]:
# Convert RFM_score into string format
rfm_df["RFM_score"] = rfm_df["RFM_score"].astype(str)

# Verify the updated data types
print(rfm_df.dtypes)

# Display sample RFM scores
print("\nSample RFM scores:")
print(rfm_df["RFM_score"].head(10))

customer_unique_id     object
recency_days            int64
frequency               int64
monetary              float64
R_score                 int64
F_score                 int64
M_score                 int64
RFM_score              object
RFM_total               int64
customer_segment       object
dtype: object

Sample RFM scores:
0    412
1    411
2    211
3    311
4    312
5    412
6    411
7    313
8    212
9    412
Name: RFM_score, dtype: object


### 4.3 Score integrity
Rebuilds `RFM_score` and `RFM_total` from the individual R, F and M scores and compares them with the stored values.

In [ ]:
# Reconstruct RFM score from individual scores
calculated_rfm = (
    rfm_df["R_score"].astype(str) +
    rfm_df["F_score"].astype(str) +
    rfm_df["M_score"].astype(str)
)

# Check mismatches
mismatches = (rfm_df["RFM_score"] != calculated_rfm).sum()

print("Total RFM score mismatches:", mismatches)

# Validate RFM total
calculated_total = (
    rfm_df["R_score"] +
    rfm_df["F_score"] +
    rfm_df["M_score"]
)

total_mismatches = (rfm_df["RFM_total"] != calculated_total).sum()

print("Total RFM total mismatches:", total_mismatches)# Reconstruct RFM score from individual scores
calculated_rfm = (
    rfm_df["R_score"].astype(str) +
    rfm_df["F_score"].astype(str) +
    rfm_df["M_score"].astype(str)
)

# Check mismatches
mismatches = (rfm_df["RFM_score"] != calculated_rfm).sum()

print("Total RFM score mismatches:", mismatches)

# Validate RFM total
calculated_total = (
    rfm_df["R_score"] +
    rfm_df["F_score"] +
    rfm_df["M_score"]
)

total_mismatches = (rfm_df["RFM_total"] != calculated_total).sum()

print("Total RFM total mismatches:", total_mismatches)

Total RFM score mismatches: 0
Total RFM total mismatches: 0
Total RFM score mismatches: 0
Total RFM total mismatches: 0


In [ ]:
# Check minimum and maximum values
print("R Score range:", rfm_df["R_score"].min(), "-", rfm_df["R_score"].max())
print("F Score range:", rfm_df["F_score"].min(), "-", rfm_df["F_score"].max())
print("M Score range:", rfm_df["M_score"].min(), "-", rfm_df["M_score"].max())

print("\nRFM Total range:",
      rfm_df["RFM_total"].min(), "-",
      rfm_df["RFM_total"].max())

# Check invalid scores
invalid_scores = rfm_df[
    (~rfm_df["R_score"].between(1, 5)) |
    (~rfm_df["F_score"].between(1, 5)) |
    (~rfm_df["M_score"].between(1, 5)) |
    (~rfm_df["RFM_total"].between(3, 15))
]

print("\nInvalid score records:", len(invalid_scores))

R Score range: 1 - 5
F Score range: 1 - 5
M Score range: 1 - 5

RFM Total range: 3 - 14

Invalid score records: 0


#### Findings: Score integrity
- 93,358 customers × 10 columns, with **no missing values and no duplicate rows**.
- R, F and M scores all lie between 1 and 5; `RFM_total` ranges from 3 to 14.
- **0 mismatches** between the stored and recomputed `RFM_score` / `RFM_total`, and 0 invalid scores.

### 4.4 Customer segment distribution

In [ ]:
# Count customers in each segment
segment_summary = (
    rfm_df["customer_segment"]
    .value_counts()
    .reset_index()
)

segment_summary.columns = ["Customer Segment", "Customer Count"]

# Calculate percentage
segment_summary["Percentage"] = (
    segment_summary["Customer Count"] /
    len(rfm_df) * 100
).round(2)

print(segment_summary)

print("\nTotal customers:", segment_summary["Customer Count"].sum())
print("Number of segments:", len(segment_summary))

      Customer Segment  Customer Count  Percentage
0      Needs Nurturing           34039       36.46
1  Potential Loyalists           20197       21.63
2          Hibernating           20008       21.43
3     Recent Customers           17904       19.18
4         Big Spenders             527        0.56
5              At Risk             490        0.52
6      Loyal Customers             184        0.20
7            Champions               9        0.01

Total customers: 93358
Number of segments: 8


#### Findings: Segment distribution
The 93,358 customers fall into **8 segments**. The largest are *Needs Nurturing* (36.46%), *Potential Loyalists* (21.63%), *Hibernating* (21.43%) and *Recent Customers* (19.18%). The remaining four segments together hold under 1.3% of customers: *Big Spenders* (0.56%), *At Risk* (0.52%), *Loyal Customers* (0.20%) and *Champions* (0.01%, 9 customers).

### 4.5 Save the validated dataset
The file below is the input for Notebook 03.

In [ ]:
# Save validated RFM dataset
rfm_df.to_csv(
    "customer_rfm_segments_clean.csv",
    index=False
)

print("Clean dataset saved successfully!")

Clean dataset saved successfully!
